# PAPER1 · Week 1 Core — Inversion Phenomenon: Layer-wise Transfer, Confound Hunt, Causal Ablation

Implements **S1–S5** of `PAPER1_MASTER_PLAN_v2.md`:

| Session | What | Output |
|---|---|---|
| S1 | Frozen SSL feature bank: wav2vec2-BASE (13 hidden states) + XLS-R-300M (25 hidden states), mean-pooled, fp16 | `features/` |
| S2 | Layer-wise logistic probes × 5 seeds, in-domain + cross-corpus | **Fig. 2** + `probe_results.json` |
| S3 | Confound hunt: per-class acoustic stats in both corpora | **Fig. 3** + `confound_stats.json` |
| S4 | Causal ablation: mel-CNN with vs without RawBoost-lite augmentation × 3 seeds | **Table 3** + `ablation_results.json` |
| S5 | Bootstrap 95% CIs + DeLong tests over everything | `stats_v1.json` |

**Setup (do once, in the Kaggle UI):**
1. Attach the In-the-Wild dataset (e.g. `bhaveshkumars/release-in-the-wild`) and your CVoiceFake dataset.
2. Point `CFG.SOURCE_*` / `CFG.TARGET_*` globs below at the right folders.
3. Accelerator: **GPU P100**. First full run ≈ 7–9 GPU-hrs. Set `CFG.DEBUG=True` for a 10-min smoke test FIRST.

The notebook is **resumable**: extracted features are cached to disk; re-running skips finished work.
All final artifacts are zipped to `/kaggle/working/week1_artifacts.zip`.

## 0 · Config — the only cell you should need to edit

In [ ]:
import os, glob, json, math, random, hashlib, warnings, time
from pathlib import Path
from dataclasses import dataclass, field

class CFG:
    # ---------------- run mode ----------------
    DEBUG            = True          # True = tiny smoke test (~10 min). Flip to False for the real run.
    SEEDS            = [0, 1, 2, 3, 4]     # probe seeds (S2/S5)
    CNN_SEEDS        = [0, 1, 2]            # mel-CNN seeds (S4)

    # ---------------- data: SOURCE domain (training corpus, e.g. CVoiceFake sample) ----
    # Point these globs at audio files. Labels: real=0 (bonafide), fake=1 (spoof).
    SOURCE_NAME      = "cvoicefake"
    SOURCE_REAL_GLOB = "/kaggle/input/YOUR-CVOICEFAKE-DATASET/**/real/**/*.wav"
    SOURCE_FAKE_GLOB = "/kaggle/input/YOUR-CVOICEFAKE-DATASET/**/fake/**/*.wav"
    SOURCE_N_PER_CLS = 10000         # 20k total source sample (plan S1)

    # ---------------- data: TARGET domain (cross-corpus eval, In-the-Wild) --------------
    TARGET_NAME      = "in_the_wild"
    TARGET_REAL_GLOB = "/kaggle/input/release-in-the-wild/**/real/**/*.wav"
    TARGET_FAKE_GLOB = "/kaggle/input/release-in-the-wild/**/fake/**/*.wav"
    TARGET_N_PER_CLS = 4000          # 8k eval, matches your v16 zero-shot n=8000

    # ---------------- audio ----------------
    SR               = 16000
    CROP_SEC         = 4.0           # crop/pad length for SSL + mel-CNN
    AUDIO_EXTS       = (".wav", ".flac", ".mp3", ".ogg")

    # ---------------- models ----------------
    SSL_MODELS = {
        "base":  "facebook/wav2vec2-base",        # 12 layers -> 13 hidden states
        "xlsr":  "facebook/wav2vec2-xls-r-300m",  # 24 layers -> 25 hidden states
    }
    SSL_BATCH        = 16            # P100-safe for 4 s clips; halve if OOM on xlsr

    # ---------------- mel-CNN (S4) ----------------
    N_MELS           = 80
    CNN_EPOCHS       = 8
    CNN_BATCH        = 64
    CNN_LR           = 3e-4

    # ---------------- stats (S5) ----------------
    N_BOOT           = 1000

    # ---------------- paths ----------------
    WORK             = Path("/kaggle/working")
    FEAT_DIR         = WORK / "features"
    FIG_DIR          = WORK / "figures"
    RES_DIR          = WORK / "results"

if CFG.DEBUG:
    CFG.SOURCE_N_PER_CLS = 150
    CFG.TARGET_N_PER_CLS = 150
    CFG.SEEDS            = [0, 1]
    CFG.CNN_SEEDS        = [0]
    CFG.CNN_EPOCHS       = 2
    CFG.N_BOOT           = 200

for d in (CFG.FEAT_DIR, CFG.FIG_DIR, CFG.RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"DEBUG={CFG.DEBUG}")

## 0.1 · Environment

In [ ]:
import subprocess, sys
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pipq("transformers>=4.40", "soundfile", "librosa")

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import librosa, soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__)
warnings.filterwarnings("ignore")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def sha1_short(s): return hashlib.sha1(s.encode()).hexdigest()[:10]

## 0.2 · Manifests — discover files, fix them with a seeded shuffle, record provenance

If your dataset layout doesn't match the real/fake glob assumption, edit `discover()` —
everything downstream only consumes the manifest lists.

In [ ]:
def discover(glob_pat, exts=CFG.AUDIO_EXTS):
    files = [f for f in glob.glob(glob_pat, recursive=True) if f.lower().endswith(exts)]
    return sorted(files)

def build_manifest(real_glob, fake_glob, n_per_cls, name, seed=1234):
    real, fake = discover(real_glob), discover(fake_glob)
    assert real and fake, (
        f"[{name}] no files found.\n  real_glob={real_glob} -> {len(real)}\n  fake_glob={fake_glob} -> {len(fake)}\n"
        "Fix the globs in CFG (check with: import glob; glob.glob(pat, recursive=True)[:5])"
    )
    rng = random.Random(seed)
    rng.shuffle(real); rng.shuffle(fake)
    real, fake = real[:n_per_cls], fake[:n_per_cls]
    paths  = real + fake
    labels = [0]*len(real) + [1]*len(fake)
    idx = list(range(len(paths))); rng.shuffle(idx)
    paths  = [paths[i] for i in idx]; labels = [labels[i] for i in idx]
    man = {"name": name, "n_real": len(real), "n_fake": len(fake),
           "paths": paths, "labels": labels}
    (CFG.RES_DIR / f"manifest_{name}.json").write_text(json.dumps(
        {k: v for k, v in man.items() if k != "paths"} | {"n_paths": len(paths)}, indent=2))
    print(f"[{name}] real={len(real)} fake={len(fake)}")
    return man

SRC = build_manifest(CFG.SOURCE_REAL_GLOB, CFG.SOURCE_FAKE_GLOB, CFG.SOURCE_N_PER_CLS, CFG.SOURCE_NAME)
TGT = build_manifest(CFG.TARGET_REAL_GLOB, CFG.TARGET_FAKE_GLOB, CFG.TARGET_N_PER_CLS, CFG.TARGET_NAME)

# in-domain split of SOURCE: 85/15 train/val (probe + CNN use the same split)
n_src = len(SRC["paths"]); cut = int(0.85 * n_src)
SRC_TRAIN = {"paths": SRC["paths"][:cut], "labels": SRC["labels"][:cut]}
SRC_VAL   = {"paths": SRC["paths"][cut:], "labels": SRC["labels"][cut:]}
print(f"source train={len(SRC_TRAIN['paths'])} val={len(SRC_VAL['paths'])} | target eval={len(TGT['paths'])}")

In [ ]:
def load_clip(path, sr=CFG.SR, crop_sec=CFG.CROP_SEC):
    """Load mono, resample, center-crop/pad to fixed length. Returns float32 [T]."""
    try:
        y, file_sr = sf.read(path, dtype="float32", always_2d=False)
        if y.ndim > 1: y = y.mean(axis=1)
        if file_sr != sr: y = librosa.resample(y, orig_sr=file_sr, target_sr=sr)
    except Exception:
        y, _ = librosa.load(path, sr=sr, mono=True)
    T = int(sr * crop_sec)
    if len(y) >= T:
        off = (len(y) - T) // 2
        y = y[off:off+T]
    else:
        y = np.pad(y, (0, T - len(y)))
    m = np.abs(y).max()
    return y / m if m > 0 else y

## S1 · Frozen SSL feature bank (resumable)

For each model × split we save one fp16 array `[N, n_layers, hidden]` of time-mean-pooled
hidden states (incl. the embedding output, so BASE→13, XLS-R→25 layers).

In [ ]:
from transformers import Wav2Vec2Model

@torch.no_grad()
def extract_features(model_key, hf_name, manifest, split_name):
    out_path = CFG.FEAT_DIR / f"{model_key}__{split_name}.npy"
    lab_path = CFG.FEAT_DIR / f"{model_key}__{split_name}__labels.npy"
    if out_path.exists() and lab_path.exists():
        print(f"[S1] cached: {out_path.name}")
        return out_path, lab_path

    print(f"[S1] extracting {model_key} on {split_name} ({len(manifest['paths'])} files)")
    model = Wav2Vec2Model.from_pretrained(hf_name).to(DEVICE).eval().half()
    feats, labs, batch, blabs = [], [], [], []
    t0 = time.time()

    def flush():
        if not batch: return
        x = torch.tensor(np.stack(batch), dtype=torch.float16, device=DEVICE)
        out = model(x, output_hidden_states=True)
        # hidden_states: tuple(n_layers+1) of [B, T', H] -> mean over time -> [B, L, H]
        hs = torch.stack([h.mean(dim=1) for h in out.hidden_states], dim=1)
        feats.append(hs.detach().cpu().numpy().astype(np.float16))
        labs.extend(blabs)
        batch.clear(); blabs.clear()

    for i, (p, l) in enumerate(zip(manifest["paths"], manifest["labels"])):
        batch.append(load_clip(p)); blabs.append(l)
        if len(batch) == CFG.SSL_BATCH: flush()
        if (i+1) % 500 == 0:
            print(f"    {i+1}/{len(manifest['paths'])}  ({time.time()-t0:.0f}s)")
    flush()

    X = np.concatenate(feats, axis=0)
    np.save(out_path, X); np.save(lab_path, np.array(labs, dtype=np.int8))
    del model; torch.cuda.empty_cache()
    print(f"[S1] saved {out_path.name}  shape={X.shape}  ({time.time()-t0:.0f}s)")
    return out_path, lab_path

SPLITS = {"src_train": SRC_TRAIN, "src_val": SRC_VAL, "tgt_eval": TGT}
FEATS = {}
for mk, hf in CFG.SSL_MODELS.items():
    for sp, man in SPLITS.items():
        FEATS[(mk, sp)] = extract_features(mk, hf, man, sp)

## S2 · Layer-wise probe sweep → **Fig. 2**

Logistic probe per layer, trained on `src_train`, scored on `src_val` (in-domain)
and `tgt_eval` (cross-corpus), for each seed. This answers:
1) which BASE layer actually transfers best (Z0 used the last layer — likely suboptimal),
2) whether XLS-R's depressed transfer holds across *all* layers, not just last/all-concat.

In [ ]:
def probe_layer(Xtr, ytr, evals, seed):
    set_seed(seed)
    scaler = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs", random_state=seed)
    clf.fit(scaler.transform(Xtr), ytr)
    out = {}
    for name, (Xe, ye) in evals.items():
        s = clf.predict_proba(scaler.transform(Xe))[:, 1]
        out[name] = {"auc": float(roc_auc_score(ye, s)), "scores": s.tolist()}
    return out

probe_results = {}   # model -> layer -> seed -> {in_domain, cross}
probe_scores_bank = {}  # for DeLong later: (model, layer, seed) -> cross scores
for mk in CFG.SSL_MODELS:
    Xtr = np.load(FEATS[(mk, "src_train")][0]).astype(np.float32)
    ytr = np.load(FEATS[(mk, "src_train")][1])
    Xva = np.load(FEATS[(mk, "src_val")][0]).astype(np.float32)
    yva = np.load(FEATS[(mk, "src_val")][1])
    Xte = np.load(FEATS[(mk, "tgt_eval")][0]).astype(np.float32)
    yte = np.load(FEATS[(mk, "tgt_eval")][1])
    L = Xtr.shape[1]
    probe_results[mk] = {}
    for layer in range(L):
        probe_results[mk][layer] = {}
        for seed in CFG.SEEDS:
            r = probe_layer(Xtr[:, layer], ytr,
                            {"in_domain": (Xva[:, layer], yva), "cross": (Xte[:, layer], yte)},
                            seed)
            probe_results[mk][layer][seed] = {k: v["auc"] for k, v in r.items()}
            probe_scores_bank[(mk, layer, seed)] = np.array(r["cross"]["scores"])
        m = np.mean([probe_results[mk][layer][s]["cross"] for s in CFG.SEEDS])
        print(f"[S2] {mk} L{layer:02d}  cross-AUC={m:.3f}")
    del Xtr, Xva, Xte

json.dump(probe_results, open(CFG.RES_DIR / "probe_results.json", "w"), indent=2, default=str)

In [ ]:
# ---- Fig. 2: layer-wise transfer curves ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for j, (mk, title) in enumerate([("base", "wav2vec2-BASE"), ("xlsr", "XLS-R 300M")]):
    layers = sorted(probe_results[mk].keys())
    for split, color in [("in_domain", "tab:gray"), ("cross", "tab:red")]:
        mu = [np.mean([probe_results[mk][l][s][split] for s in CFG.SEEDS]) for l in layers]
        sd = [np.std ([probe_results[mk][l][s][split] for s in CFG.SEEDS]) for l in layers]
        ax[j].plot(layers, mu, "-o", ms=3, color=color, label=split)
        ax[j].fill_between(layers, np.array(mu)-np.array(sd), np.array(mu)+np.array(sd),
                           color=color, alpha=0.2)
    ax[j].axhline(0.5, ls="--", c="k", lw=0.8)
    ax[j].set_title(title); ax[j].set_xlabel("hidden-state index"); ax[j].grid(alpha=0.3)
ax[0].set_ylabel("ROC-AUC"); ax[0].legend()
fig.suptitle("Fig. 2 — Layer-wise transfer: in-domain vs cross-corpus")
fig.tight_layout(); fig.savefig(CFG.FIG_DIR / "fig2_layerwise_transfer.png", dpi=200)
plt.close(fig); print("saved fig2")

best = {mk: max(probe_results[mk], key=lambda l: np.mean([probe_results[mk][l][s]["cross"] for s in CFG.SEEDS]))
        for mk in probe_results}
print("best cross-corpus layer per model:", best)

## S3 · Confound hunt → **Fig. 3**

Hypothesis: the mel-CNN's AUC 0.178 is an *anti-correlated channel shortcut* —
in the target corpus, fakes are wider-band / cleaner than reals (opposite of source).
We measure per-class: spectral rolloff (95%), estimated bandwidth cutoff, spectral
centroid, RMS, silence ratio, spectral flatness.

In [ ]:
def acoustic_stats(path):
    y = load_clip(path)
    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256)) + 1e-10
    freqs = librosa.fft_frequencies(sr=CFG.SR, n_fft=1024)
    rolloff  = float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=CFG.SR, roll_percent=0.95)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(S=S, sr=CFG.SR)))
    flatness = float(np.mean(librosa.feature.spectral_flatness(S=S)))
    rms      = float(np.mean(librosa.feature.rms(S=S)))
    # crude bandwidth cutoff: highest freq bin whose long-term energy is within 40 dB of peak band
    band_db  = 20*np.log10(S.mean(axis=1)); band_db -= band_db.max()
    above    = np.where(band_db > -40)[0]
    cutoff   = float(freqs[above[-1]]) if len(above) else 0.0
    sil      = float(np.mean(librosa.feature.rms(y=y, frame_length=1024, hop_length=256)[0] < 0.01))
    return dict(rolloff=rolloff, centroid=centroid, flatness=flatness,
                rms=rms, cutoff=cutoff, silence_ratio=sil)

def corpus_stats(man, name, cap=1500):
    rng = random.Random(7)
    idx = list(range(len(man["paths"]))); rng.shuffle(idx); idx = idx[:cap]
    rows = []
    for k, i in enumerate(idx):
        st = acoustic_stats(man["paths"][i]); st["label"] = man["labels"][i]; rows.append(st)
        if (k+1) % 300 == 0: print(f"    [{name}] {k+1}/{len(idx)}")
    return rows

print("[S3] source stats"); src_stats = corpus_stats({"paths": SRC["paths"], "labels": SRC["labels"]}, "src")
print("[S3] target stats"); tgt_stats = corpus_stats(TGT, "tgt")
json.dump({"source": src_stats, "target": tgt_stats},
          open(CFG.RES_DIR / "confound_stats.json", "w"))

In [ ]:
# ---- Fig. 3: per-class distributions, source vs target ----
KEYS = ["cutoff", "rolloff", "flatness", "silence_ratio"]
fig, axes = plt.subplots(2, len(KEYS), figsize=(4*len(KEYS), 7), sharex="col")
for row, (stats, cname) in enumerate([(src_stats, f"SOURCE ({CFG.SOURCE_NAME})"),
                                      (tgt_stats, f"TARGET ({CFG.TARGET_NAME})")]):
    for col, key in enumerate(KEYS):
        for lab, color, nm in [(0, "tab:blue", "real"), (1, "tab:orange", "fake")]:
            vals = [r[key] for r in stats if r["label"] == lab]
            axes[row][col].hist(vals, bins=40, alpha=0.55, color=color, label=nm, density=True)
        axes[row][col].set_title(f"{cname}\n{key}" if row == 0 else key, fontsize=9)
        axes[row][col].grid(alpha=0.3)
axes[0][0].legend()
fig.suptitle("Fig. 3 — Per-class acoustic confounds: does the real/fake channel gap FLIP across corpora?")
fig.tight_layout(); fig.savefig(CFG.FIG_DIR / "fig3_confounds.png", dpi=200); plt.close(fig)

# quick numeric summary: class-mean gap per corpus (sign flip = shortcut smoking gun)
summary = {}
for key in KEYS:
    def gap(stats):
        f = np.mean([r[key] for r in stats if r["label"] == 1])
        r_ = np.mean([r[key] for r in stats if r["label"] == 0])
        return f - r_
    summary[key] = {"source_gap(fake-real)": gap(src_stats), "target_gap(fake-real)": gap(tgt_stats),
                    "sign_flip": bool(np.sign(gap(src_stats)) != np.sign(gap(tgt_stats)))}
json.dump(summary, open(CFG.RES_DIR / "confound_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

## S4 · Causal ablation: mel-CNN ± RawBoost-lite → **Table 3**

Prediction: augmentation destroys the channel shortcut → cross-AUC moves 0.18 → ~0.5,
while in-domain drops only mildly. (If instead cross-AUC *rises* meaningfully above 0.5,
the augmented CNN learned real artifacts — also reportable.)

In [ ]:
def rawboost_lite(y, rng):
    """Lightweight stand-in for RawBoost: additive noise + random low-pass + clipping + gain.
    Purpose: destroy channel/bandwidth shortcuts, not to reproduce RawBoost exactly."""
    y = y.copy()
    if rng.random() < 0.8:                                   # colored noise
        snr_db = rng.uniform(8, 30)
        noise = np.random.default_rng(rng.randint(0, 1<<30)).standard_normal(len(y)).astype(np.float32)
        b = np.array([1.0, rng.uniform(-0.9, 0.9)], dtype=np.float32)   # 1st-order color
        noise = np.convolve(noise, b, mode="same")
        p_sig = np.mean(y**2) + 1e-12; p_noi = np.mean(noise**2) + 1e-12
        noise *= math.sqrt(p_sig / p_noi / (10 ** (snr_db / 10)))
        y = y + noise
    if rng.random() < 0.7:                                   # random band-limit
        cutoff = rng.uniform(3000, 7600)
        n_fft = 1024
        Y = librosa.stft(y, n_fft=n_fft, hop_length=256)
        freqs = librosa.fft_frequencies(sr=CFG.SR, n_fft=n_fft)
        Y[freqs > cutoff, :] *= rng.uniform(0.0, 0.1)
        y = librosa.istft(Y, hop_length=256, length=len(y))
    if rng.random() < 0.3:                                   # clipping
        thr = rng.uniform(0.6, 0.95); y = np.clip(y, -thr, thr)
    y *= rng.uniform(0.5, 1.0)                               # gain
    m = np.abs(y).max();  return (y / m if m > 0 else y).astype(np.float32)

MEL = librosa.filters.mel(sr=CFG.SR, n_fft=1024, n_mels=CFG.N_MELS)
def logmel(y):
    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))**2
    M = np.log(MEL @ S + 1e-8)
    return ((M - M.mean()) / (M.std() + 1e-6)).astype(np.float32)

class MelDS(torch.utils.data.Dataset):
    def __init__(self, man, augment=False, seed=0):
        self.paths, self.labels = man["paths"], man["labels"]
        self.augment, self.seed = augment, seed
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        y = load_clip(self.paths[i])
        if self.augment:
            y = rawboost_lite(y, random.Random((self.seed * 1_000_003 + i)))
        return torch.from_numpy(logmel(y))[None], self.labels[i]

class MelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        ch = [1, 32, 64, 128, 128]
        self.blocks = nn.Sequential(*[
            nn.Sequential(nn.Conv2d(ch[i], ch[i+1], 3, padding=1), nn.BatchNorm2d(ch[i+1]),
                          nn.ReLU(), nn.MaxPool2d(2)) for i in range(4)])
        self.head = nn.Linear(128, 1)
    def forward(self, x):
        h = self.blocks(x).mean(dim=(2, 3))
        return self.head(h).squeeze(-1)

@torch.no_grad()
def cnn_scores(model, man):
    model.eval()
    dl = torch.utils.data.DataLoader(MelDS(man), batch_size=CFG.CNN_BATCH, num_workers=2)
    ss, yy = [], []
    for x, y in dl:
        ss.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()); yy.append(y.numpy())
    return np.concatenate(ss), np.concatenate(yy)

def train_cnn(augment, seed):
    set_seed(seed)
    model = MelCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.CNN_LR)
    dl = torch.utils.data.DataLoader(MelDS(SRC_TRAIN, augment=augment, seed=seed),
                                     batch_size=CFG.CNN_BATCH, shuffle=True, num_workers=2, drop_last=True)
    for ep in range(CFG.CNN_EPOCHS):
        model.train(); tot = 0.0
        for x, y in dl:
            x, y = x.to(DEVICE), y.float().to(DEVICE)
            loss = F.binary_cross_entropy_with_logits(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        print(f"    [aug={augment} seed={seed}] ep{ep+1}/{CFG.CNN_EPOCHS} loss={tot/len(dl):.4f}")
    s_val, y_val = cnn_scores(model, SRC_VAL)
    s_tgt, y_tgt = cnn_scores(model, TGT)
    return {"in_domain_auc": float(roc_auc_score(y_val, s_val)),
            "cross_auc":     float(roc_auc_score(y_tgt, s_tgt)),
            "cross_scores":  s_tgt.tolist(), "cross_labels": y_tgt.tolist()}

ablation = {"no_aug": {}, "rawboost_lite": {}}
for seed in CFG.CNN_SEEDS:
    print(f"[S4] mel-CNN no_aug seed={seed}");        ablation["no_aug"][seed]        = train_cnn(False, seed)
    print(f"[S4] mel-CNN rawboost_lite seed={seed}"); ablation["rawboost_lite"][seed] = train_cnn(True,  seed)

table3 = {cond: {
    "in_domain_auc_mean": float(np.mean([r["in_domain_auc"] for r in runs.values()])),
    "in_domain_auc_std":  float(np.std ([r["in_domain_auc"] for r in runs.values()])),
    "cross_auc_mean":     float(np.mean([r["cross_auc"]     for r in runs.values()])),
    "cross_auc_std":      float(np.std ([r["cross_auc"]     for r in runs.values()])),
} for cond, runs in ablation.items()}
json.dump({"per_run": ablation, "table3": table3},
          open(CFG.RES_DIR / "ablation_results.json", "w"), indent=2)
print("[S4] Table 3:", json.dumps(table3, indent=2))

## S5 · Statistics pass — bootstrap CIs + DeLong tests → `stats_v1.json`

In [ ]:
def bootstrap_auc_ci(y, s, n_boot=CFG.N_BOOT, seed=0):
    rng = np.random.default_rng(seed); y, s = np.asarray(y), np.asarray(s)
    aucs = []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2: continue
        aucs.append(roc_auc_score(y[idx], s[idx]))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

def delong_test(y, s1, s2):
    """Fast DeLong two-sided p-value for correlated ROC AUCs (Sun & Xu 2014 midrank method)."""
    y = np.asarray(y); order = np.argsort(-y)          # positives first
    y = y[order]; m = int(y.sum()); n = len(y) - m
    def midrank(x):
        J = np.argsort(x); Z = x[J]; N = len(x)
        T = np.zeros(N); i = 0
        while i < N:
            j = i
            while j < N and Z[j] == Z[i]: j += 1
            T[i:j] = 0.5 * (i + j - 1) + 1; i = j
        out = np.empty(N); out[J] = T; return out
    preds = np.vstack([np.asarray(s1)[order], np.asarray(s2)[order]])
    k = 2
    tx = np.array([midrank(preds[r, :m]) for r in range(k)])
    ty = np.array([midrank(preds[r, m:]) for r in range(k)])
    tz = np.array([midrank(preds[r, :])  for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx, sy = np.cov(v01), np.cov(v10)
    S = sx / m + sy / n
    d = aucs[0] - aucs[1]
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var <= 0: return float(aucs[0]), float(aucs[1]), 1.0
    z = d / math.sqrt(var)
    from scipy.stats import norm
    return float(aucs[0]), float(aucs[1]), float(2 * norm.sf(abs(z)))

yte = np.load(FEATS[("base", "tgt_eval")][1])
stats_out = {"config": {"debug": CFG.DEBUG, "seeds": CFG.SEEDS, "n_boot": CFG.N_BOOT},
             "layerwise_ci": {}, "delong": {}}

# CIs for best layer of each model (seed 0 scores)
for mk in CFG.SSL_MODELS:
    bl = best[mk]; s = probe_scores_bank[(mk, bl, CFG.SEEDS[0])]
    lo, hi = bootstrap_auc_ci(yte, s)
    stats_out["layerwise_ci"][mk] = {"best_layer": int(bl),
        "cross_auc": float(roc_auc_score(yte, s)), "ci95": [lo, hi]}

# DeLong: best BASE layer vs best XLS-R layer (the 'bigger transfers worse' claim)
a1, a2, p = delong_test(yte, probe_scores_bank[("base", best["base"], CFG.SEEDS[0])],
                              probe_scores_bank[("xlsr", best["xlsr"], CFG.SEEDS[0])])
stats_out["delong"]["base_best_vs_xlsr_best"] = {"auc_base": a1, "auc_xlsr": a2, "p": p}

# DeLong: SSL best-layer probe vs mel-CNN (no_aug, seed0) — the inversion headline
r0 = ablation["no_aug"][CFG.CNN_SEEDS[0]]
a1, a2, p = delong_test(np.array(r0["cross_labels"]),
                        probe_scores_bank[("base", best["base"], CFG.SEEDS[0])][:len(r0["cross_labels"])],
                        np.array(r0["cross_scores"]))
stats_out["delong"]["ssl_vs_melcnn_cross"] = {"auc_ssl": a1, "auc_melcnn": a2, "p": p}

# CIs for Table 3 conditions (seed 0)
for cond in ablation:
    r = ablation[cond][CFG.CNN_SEEDS[0]]
    lo, hi = bootstrap_auc_ci(np.array(r["cross_labels"]), np.array(r["cross_scores"]))
    stats_out.setdefault("melcnn_ci", {})[cond] = {"cross_auc": r["cross_auc"], "ci95": [lo, hi]}

json.dump(stats_out, open(CFG.RES_DIR / "stats_v1.json", "w"), indent=2)
print(json.dumps(stats_out, indent=2))

## Wrap-up — zip artifacts, print the week-1 verdict

In [ ]:
import shutil
stage = CFG.WORK / "_stage"; stage.mkdir(exist_ok=True)
for d in ("figures", "results"):
    shutil.copytree(CFG.WORK / d, stage / d, dirs_exist_ok=True)
shutil.make_archive(str(CFG.WORK / "week1_artifacts"), "zip", root_dir=stage)
shutil.rmtree(stage)  # features/ intentionally NOT zipped — save them as a Kaggle dataset instead

print("="*70)
print("WEEK 1 VERDICT (paste into PAPER1 notes)")
print("="*70)
for mk in CFG.SSL_MODELS:
    c = stats_out["layerwise_ci"][mk]
    print(f"  {mk:5s} best layer L{c['best_layer']:02d}: cross-AUC {c['cross_auc']:.3f} CI{c['ci95']}")
print(f"  DeLong base-vs-xlsr p = {stats_out['delong']['base_best_vs_xlsr_best']['p']:.2e}")
print(f"  mel-CNN no_aug cross-AUC   = {table3['no_aug']['cross_auc_mean']:.3f} ± {table3['no_aug']['cross_auc_std']:.3f}")
print(f"  mel-CNN rawboost cross-AUC = {table3['rawboost_lite']['cross_auc_mean']:.3f} ± {table3['rawboost_lite']['cross_auc_std']:.3f}")
flips = [k for k, v in summary.items() if v["sign_flip"]]
print(f"  confound sign-flips: {flips if flips else 'NONE — shortcut hypothesis weakened, report honestly'}")
print("  interpretation: aug moving cross-AUC toward 0.5 while flips exist => channel-shortcut CONFIRMED")
print("="*70)
print("Artifacts: /kaggle/working/week1_artifacts.zip  |  features cached in /kaggle/working/features/")
print("Next: save features/ as a Kaggle dataset 'features_bank_v1' so S6+ notebooks reuse it for free.")